In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import random 

import time 

root_dir = os.path.abspath('../..')
if root_dir not in sys.path:
    sys.path.append(root_dir)


from src.data_loader.data_module_lightning import SonarSimDataModule

from src.models.graph_train import Graph 


In [10]:
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'
train_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/training.yaml'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(model_config_pth, "r") as f:
    model_config = Box(yaml.safe_load(f))

with open(sonar_config_pth, "r") as f:
    sonar_config = Box(yaml.safe_load(f))



graph = Graph(model_config, sonar_config, batch_size = 1, frames_in_series = 2)

            
root_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data'
batch_size = 2
num_workers = 0
transforms = None
dm = SonarSimDataModule(root_dir, batch_size, num_workers, transforms, train_config_pth)
dm.setup(stage ='fit')
train_data_loader = dm.train_dataloader()

batch = next(iter(train_data_loader))
frames, time, trajectory, depth = batch

print('Input data format')
print(f'time: {time.shape}')
print(f'fls frame: {frames.shape}')
print(f'pose: {trajectory.shape}')
print(f'depth: {depth.shape}')



TypeError: '>' not supported between instances of 'int' and 'str'

## 1. Append graph 

- Pass data through patchifier, get: patches features, key points coords
- Init pose (based on initial pose or by movement approximation)
- Init elevation angle 
- Create Poses Nodes in Graph 
- Create Patches Nodes in Graph
- Create Edges between Poses and Patches Nodes

## 2. Update graph 
- Clculate correlation between patches in source frames and patches projected on target frames
- Edges validation - discard patches that are projected out of range on sonar 


In [ ]:
# Training Graph 
with torch.no_grad(): # to avoid build torch graph in tests
    # append graph
    poses, coords_phi = graph_t.append(frames, time, trajectory, device)
    # update step
    corr, ctx, source_frame_idx, target_frame_idx, patch_idx = graph_t.update_step(poses, coords_phi, device)

print('Graph append results:')
print(f'Poses nodes number: {poses.shape[0]*poses.shape[1]} (shape: {poses.shape})')
print(f'Patch nodes created: {coords_phi.shape[0]*coords_phi.shape[1]*coords_phi.shape[2]} (shape: {coords_phi.shape})')

print('Graph update results:')
print(f'correlation: {corr.shape}')
print(f'ctx: {ctx.shape}')
print(f'source frames: {source_frame_idx.shape}')
print(f'source frames: {target_frame_idx.shape}')
print(f'source frames: {patch_idx.shape}')


Graph append results:
Poses nodes number: 10 (shape: torch.Size([2, 5, 7]))
Patch nodes created: 50 (shape: torch.Size([2, 5, 5, 1]))
Graph update results:
correlation: torch.Size([4, 50])
ctx: torch.Size([4, 384])
source frames: torch.Size([4])
source frames: torch.Size([4])
source frames: torch.Size([4])


In [ ]:
# Inference graph
with torch.no_grad():
    for b in range(time.shape[0]): # iterate through batch
        for n in range(time.shape[1]):
            frame_ = frames[b, n, ...]
            time_ = time[b, n]

            graph_i.append(frame_.unsqueeze(0).unsqueeze(0), time_.unsqueeze(0).unsqueeze(0), device)

            corr, ctx, source_frame_idx, target_frame_idx, patch_idx = graph_i.update_step(device)

    print('Graph append results')
    print(f'Poses nodes number: {graph_i.frame_n}')
    print(f'Patch nodes created: {graph_i.frame_n * graph_i.patches_per_frame}')

    print('Graph update results:')
    print(f'correlation: {corr.shape}')
    print(f'ctx: {ctx.shape}')
    print(f'source frames: {source_frame_idx.shape}')
    print(f'source frames: {target_frame_idx.shape}')
    print(f'source frames: {patch_idx.shape}')

Graph append results
Poses nodes number: 10
Patch nodes created: 50
Graph update results:
correlation: torch.Size([15, 50])
ctx: torch.Size([15, 384])
source frames: torch.Size([15])
source frames: torch.Size([15])
source frames: torch.Size([15])


<!-- - prepare data  -->

<!-- ## Output test -->
